# Lung tumour outlining — MedSAM 2 vs our engine

**What this notebook does:** runs a modern AI model (MedSAM 2) on the *same exam* as our current engine,
and compares both with the expert outlines.

- **Data:** the public RIDER Lung CT scans and expert outlines (RIDER-LungCT-Seg), downloaded straight from TCIA.
- **The exam:** for every scan the tool gets one box around the tumour on the expert's widest slice,
  and must outline the whole tumour in 3D. We then compare with the expert.
- **Models tested:** `MedSAM2_latest` (general medical) and `MedSAM2_CTLesion` (trained on CT lesions).

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
It takes about 20–40 minutes for all 59 scans (most of it is downloading). Set `N_CASES` below to try fewer first.

*Research use only. Public, de-identified data. No clinical claims.*

In [ ]:
N_CASES = 59          # 59 = all expert-outlined scans. Use e.g. 6 for a quick first run.
MODELS = ["MedSAM2_latest.pt", "MedSAM2_CTLesion.pt"]
WINDOW = (-1350, 150)  # lung window used by DeepLesion / MedSAM2 for lung lesions

import subprocess, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU!")

## 1. Install MedSAM 2 and download the model weights (from Hugging Face)

In [ ]:
%cd /content
!git clone -q --depth 1 https://github.com/bowang-lab/MedSAM2.git
%cd /content/MedSAM2
!SAM2_BUILD_CUDA=0 pip install -q --no-deps -e .
!pip install -q hydra-core iopath pydicom SimpleITK
!mkdir -p checkpoints
for m in MODELS:
    subprocess.run(["wget", "-q", "-nc", "-P", "checkpoints", f"https://huggingface.co/wanglab/MedSAM2/resolve/main/{m}"], check=True)
!ls -la checkpoints

## 2. The 59 scans of the exam, and our engine's scores (copied from the project)

In [ ]:
cases = [["RIDER-1129164940", "TEST", "1.2.276.0.7230010.3.1.3.8323329.2021.1554822307.24503", "1.3.6.1.4.1.9328.50.1.83304264089411327530730818890072724533"], ["RIDER-1129164940", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.20780.1554842143.193341", "1.3.6.1.4.1.9328.50.1.48441840081578419409180519840073808100"], ["RIDER-1225316081", "TEST", "1.2.276.0.7230010.3.1.3.8323329.5678.1554820235.472211", "1.3.6.1.4.1.9328.50.1.136644362331225154777907863722323747874"], ["RIDER-1225316081", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.7091.1554843537.973188", "1.3.6.1.4.1.9328.50.1.57875588819948681104622543575484490295"], ["RIDER-1286684383", "TEST", "1.2.276.0.7230010.3.1.3.8323329.18140.1554821167.472678", "1.3.6.1.4.1.9328.50.1.250276730034374548755125887754378560209"], ["RIDER-1286684383", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.10537.1554843803.471316", "1.3.6.1.4.1.9328.50.1.54869681489123387741768260322440045297"], ["RIDER-1332496276", "TEST", "1.2.276.0.7230010.3.1.3.8323329.3432.1554822400.109841", "1.3.6.1.4.1.9328.50.1.42697596859477567872763647333745089432"], ["RIDER-1332496276", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.18239.1554844358.919088", "1.3.6.1.4.1.9328.50.1.51182298817488926005730686623574691733"], ["RIDER-1344011628", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.3483.1554843262.971275", "1.3.6.1.4.1.9328.50.1.66108458465186904026908357142054909583"], ["RIDER-1344011628", "TEST", "1.2.276.0.7230010.3.1.3.8323329.23231.1554821525.429266", "1.3.6.1.4.1.9328.50.1.70885448642275408073484639858923588464"], ["RIDER-1346917136", "TEST", "1.2.276.0.7230010.3.1.3.8323329.31653.1554822125.805873", "1.3.6.1.4.1.9328.50.1.298748100008818908989408006815250397386"], ["RIDER-1346917136", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.16870.1554844272.214723", "1.3.6.1.4.1.9328.50.1.241687561050325650322607529919565760760"], ["RIDER-1500037140", "TEST", "1.2.276.0.7230010.3.1.3.8323329.6703.1554820324.285935", "1.3.6.1.4.1.9328.50.1.275712112233733729527949643353021665322"], ["RIDER-1500037140", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.11604.1554843904.94647", "1.3.6.1.4.1.9328.50.1.228849430732466946515663085966276560854"], ["RIDER-1532432635", "TEST", "1.2.276.0.7230010.3.1.3.8323329.3151.1554820054.833742", "1.3.6.1.4.1.9328.50.1.244501235124778437799277943012383090930"], ["RIDER-1532432635", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.23409.1554842387.560675", "1.3.6.1.4.1.9328.50.1.251497901416717723949588736705581667952"], ["RIDER-1760553574", "TEST", "1.2.276.0.7230010.3.1.3.8323329.4630.1554822493.511219", "1.3.6.1.4.1.9328.50.1.327399981350769389671009921076478636105"], ["RIDER-1760553574", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.9236.1554843710.400783", "1.3.6.1.4.1.9328.50.1.217618293370677618119893300068426363153"], ["RIDER-1801720707", "TEST", "1.2.276.0.7230010.3.1.3.8323329.25780.1554821702.493501", "1.3.6.1.4.1.9328.50.1.269982481234503399884470277848651506493"], ["RIDER-1801720707", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.26032.1554842575.927371", "1.3.6.1.4.1.9328.50.1.234211180786790460829763655261513482302"], ["RIDER-1825099523", "TEST", "1.2.276.0.7230010.3.1.3.8323329.15604.1554820995.783866", "1.3.6.1.4.1.9328.50.1.151205757722151678647268859397211234394"], ["RIDER-1825099523", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.12950.1554844002.252553", "1.3.6.1.4.1.9328.50.1.277306877703927900763621250092700839081"], ["RIDER-1867199702", "TEST", "1.2.276.0.7230010.3.1.3.8323329.1705.1554819982.390085", "1.3.6.1.4.1.9328.50.1.159175208197607220484095971384563377994"], ["RIDER-1867199702", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.2204.1554843187.867545", "1.3.6.1.4.1.9328.50.1.176840664438918409569682530893469371660"], ["RIDER-2016615262", "TEST", "1.2.276.0.7230010.3.1.3.8323329.4480.1554820147.296719", "1.3.6.1.4.1.9328.50.1.299673360876570070700703250582150584718"], ["RIDER-2016615262", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.11318.1556878233.850552", "1.3.6.1.4.1.9328.50.1.211449458002306195716642985423685291150"], ["RIDER-2073083717", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.15621.1554844195.560572", "1.3.6.1.4.1.9328.50.1.37531487280017801124765643732079524443"], ["RIDER-2073083717", "TEST", "1.2.276.0.7230010.3.1.3.8323329.29296.1554821950.541699", "1.3.6.1.4.1.9328.50.1.205940897977851990746556006144481623627"], ["RIDER-2151469008", "TEST", "1.2.276.0.7230010.3.1.3.8323329.16806.1554821083.229719", "1.3.6.1.4.1.9328.50.1.94933979665959337442766748630370536812"], ["RIDER-2151469008", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.22045.1554842223.669661", "1.3.6.1.4.1.9328.50.1.284014020710168432486041624571005092281"], ["RIDER-2283289298", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.14148.1554844092.679599", "1.3.6.1.4.1.9328.50.1.109288921990958367956324078058642091696"], ["RIDER-2357766186", "TEST", "1.2.276.0.7230010.3.1.3.8323329.14205.1554820902.237181", "1.3.6.1.4.1.9328.50.1.245630263591502535745452645381329674063"], ["RIDER-2357766186", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.30717.1554842934.942943", "1.3.6.1.4.1.9328.50.1.160525591228102999616019562758104412505"], ["RIDER-2539508879", "TEST", "1.2.276.0.7230010.3.1.3.8323329.10399.1554820614.125440", "1.3.6.1.4.1.9328.50.1.158120423511327812902957049440486097165"], ["RIDER-2539508879", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.19578.1554842047.820136", "1.3.6.1.4.1.9328.50.1.215800995114148697334431058444392949526"], ["RIDER-2541949645", "TEST", "1.2.276.0.7230010.3.1.3.8323329.20685.1554821352.112310", "1.3.6.1.4.1.9328.50.1.152572901056058406211409536989510187742"], ["RIDER-2541949645", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.8070.1554843626.111330", "1.3.6.1.4.1.9328.50.1.84023861072193772859482594252670063850"], ["RIDER-2619750334", "TEST", "1.2.276.0.7230010.3.1.3.8323329.12891.1554820797.597451", "1.3.6.1.4.1.9328.50.1.307319453402340384126936296040122224868"], ["RIDER-2619750334", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.26932.1554842656.958509", "1.3.6.1.4.1.9328.50.1.235386105589494260919744152317749907915"], ["RIDER-2655999012", "TEST", "1.2.276.0.7230010.3.1.3.8323329.364.1554819884.486584", "1.3.6.1.4.1.9328.50.1.112793496676844175431872842334447612042"], ["RIDER-2655999012", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.18411.1554841949.273338", "1.3.6.1.4.1.9328.50.1.5804567971224333682982299447203570095"], ["RIDER-2669524182", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.28168.1554842757.156486", "1.3.6.1.4.1.9328.50.1.269463014648045535201701159778694569334"], ["RIDER-2669524182", "TEST", "1.2.276.0.7230010.3.1.3.8323329.21953.1554821436.288515", "1.3.6.1.4.1.9328.50.1.164139597747545377343600469399472836968"], ["RIDER-2799584460", "TEST", "1.2.276.0.7230010.3.1.3.8323329.591.1554822215.562011", "1.3.6.1.4.1.9328.50.1.182276903811735605390981624025960600655"], ["RIDER-2799584460", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.31907.1554843015.400512", "1.3.6.1.4.1.9328.50.1.292474823408043076724467505606424288322"], ["RIDER-3023568408", "TEST", "1.2.276.0.7230010.3.1.3.8323329.19345.1554821263.444952", "1.3.6.1.4.1.9328.50.1.177903673228104341213224928761701786693"], ["RIDER-3023568408", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.24761.1554842482.122464", "1.3.6.1.4.1.9328.50.1.76808802982262101679502232663993705555"], ["RIDER-3115188676", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.747.1554843102.753472", "1.3.6.1.4.1.9328.50.1.136792069117584747719409894247257396682"], ["RIDER-3115188676", "TEST", "1.2.276.0.7230010.3.1.3.8323329.7982.1554820417.354881", "1.3.6.1.4.1.9328.50.1.255647697326259514897843794470928560217"], ["RIDER-3152132495", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.19480.1554844461.437970", "1.3.6.1.4.1.9328.50.1.334215469883390884437229836705492234507"], ["RIDER-3152132495", "TEST", "1.2.276.0.7230010.3.1.3.8323329.28010.1554821863.449566", "1.3.6.1.4.1.9328.50.1.116324576791139455082825091960790224867"], ["RIDER-3160137230", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.4613.1554843349.990449", "1.3.6.1.4.1.9328.50.1.259840310625640684686783854863615824641"], ["RIDER-3160137230", "TEST", "1.2.276.0.7230010.3.1.3.8323329.26750.1554821783.648863", "1.3.6.1.4.1.9328.50.1.191985769478320383635122902894438276122"], ["RIDER-5195703382", "TEST", "1.2.276.0.7230010.3.1.3.8323329.11554.1554820706.857013", "1.3.6.1.4.1.9328.50.1.286170493057098735157197376891061719104"], ["RIDER-6324870494", "TEST", "1.2.276.0.7230010.3.1.3.8323329.24558.1554821613.346079", "1.3.6.1.4.1.9328.50.1.52136035328653293207079755735387340761"], ["RIDER-6324870494", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.29519.1554842848.297024", "1.3.6.1.4.1.9328.50.1.286680119667247381026746579579720294047"], ["RIDER-8509201188", "TEST", "1.2.276.0.7230010.3.1.3.8323329.9112.1554820506.221187", "1.3.6.1.4.1.9328.50.1.232408293761209220974761841467324190361"], ["RIDER-9763310455", "TEST", "1.2.276.0.7230010.3.1.3.8323329.30503.1554822027.916239", "1.3.6.1.4.1.9328.50.1.153467821594768929811844380798337401116"], ["RIDER-9763310455", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.5891.1554843444.209653", "1.3.6.1.4.1.9328.50.1.70949788529300490780576543460245043945"]]
cases = cases[:N_CASES]
print(len(cases), 'scans')

V5_CSV = '''patient,scan,dice,expert_ml,tool_ml,expert_mm,tool_mm,seconds
RIDER-1129164940,RETEST,0.879,54.47,57.09,60.2,62.0,53.63
RIDER-1129164940,TEST,0.871,54.99,59.21,59.0,62.0,56.61
RIDER-1225316081,RETEST,0.643,0.56,0.35,13.3,13.7,16.39
RIDER-1225316081,TEST,0.726,0.64,0.47,14.0,15.5,16.2
RIDER-1286684383,RETEST,0.597,4.18,6.37,38.2,38.6,39.96
RIDER-1286684383,TEST,0.616,7.15,10.58,51.2,55.4,4.97
RIDER-1332496276,RETEST,0.843,25.05,21.62,47.6,40.0,45.62
RIDER-1332496276,TEST,0.722,27.76,33.23,51.0,47.9,43.9
RIDER-1344011628,RETEST,0.52,31.38,70.85,63.1,69.9,65.01
RIDER-1344011628,TEST,0.455,38.7,99.6,61.1,73.2,63.0
RIDER-1346917136,RETEST,0.973,88.27,84.47,65.2,63.0,73.98
RIDER-1346917136,TEST,0.951,90.49,84.44,63.3,63.1,66.34
RIDER-1500037140,RETEST,0.865,10.31,7.88,32.2,29.5,36.07
RIDER-1500037140,TEST,0.889,8.84,7.23,30.1,29.7,34.56
RIDER-1532432635,RETEST,0.667,37.2,67.29,51.2,59.8,52.29
RIDER-1532432635,TEST,0.705,57.13,74.24,65.0,69.4,63.54
RIDER-1760553574,RETEST,0.382,3.91,1.75,30.8,22.3,34.29
RIDER-1760553574,TEST,0.518,4.61,2.71,24.8,23.9,28.35
RIDER-1801720707,RETEST,0.599,0.29,0.13,12.6,8.9,15.32
RIDER-1801720707,TEST,0.382,0.55,0.13,12.8,9.6,15.63
RIDER-1825099523,RETEST,0.951,24.32,22.67,44.1,49.0,48.02
RIDER-1825099523,TEST,0.928,25.17,21.92,41.9,40.9,45.82
RIDER-1867199702,RETEST,0.849,1.63,1.41,16.9,15.5,20.91
RIDER-1867199702,TEST,0.849,1.92,1.43,17.9,15.7,20.5
RIDER-2016615262,RETEST,0.463,29.4,90.4,49.0,59.4,48.22
RIDER-2016615262,TEST,0.445,59.31,166.81,66.6,78.1,71.61
RIDER-2073083717,RETEST,0.456,5.02,16.52,28.6,35.1,35.95
RIDER-2073083717,TEST,0.421,6.11,21.0,33.1,40.3,42.73
RIDER-2151469008,RETEST,0.71,40.59,66.36,52.9,59.9,60.76
RIDER-2151469008,TEST,0.643,54.17,87.13,56.9,63.4,67.44
RIDER-2283289298,RETEST,0.807,8.62,5.84,31.9,29.0,36.86
RIDER-2357766186,RETEST,0.783,22.85,16.29,58.0,55.1,65.56
RIDER-2357766186,TEST,0.807,18.24,16.1,49.9,54.0,56.37
RIDER-2539508879,RETEST,0.915,9.07,7.93,35.3,36.3,42.13
RIDER-2539508879,TEST,0.861,9.92,7.62,37.5,39.2,44.65
RIDER-2541949645,RETEST,0.052,1.94,0.05,18.1,9.1,19.93
RIDER-2541949645,TEST,0.386,1.79,0.57,19.0,17.2,21.47
RIDER-2619750334,RETEST,0.815,86.58,110.47,76.4,84.8,69.85
RIDER-2619750334,TEST,0.708,91.96,148.42,83.4,99.3,9.45
RIDER-2655999012,RETEST,0.697,2.77,2.62,28.6,32.4,30.48
RIDER-2655999012,TEST,0.811,3.51,3.53,35.9,36.7,35.63
RIDER-2669524182,RETEST,0.504,23.11,7.84,41.2,38.2,42.5
RIDER-2669524182,TEST,0.76,17.78,15.63,41.9,36.0,39.35
RIDER-2799584460,RETEST,0.705,0.77,0.42,12.2,9.7,14.76
RIDER-2799584460,TEST,0.67,0.89,0.45,12.6,10.0,14.86
RIDER-3023568408,RETEST,0.904,4.89,4.44,23.5,23.5,23.67
RIDER-3023568408,TEST,0.792,6.41,4.3,25.8,25.3,25.62
RIDER-3115188676,RETEST,0.699,12.75,6.98,48.9,41.1,48.44
RIDER-3115188676,TEST,0.806,10.35,7.14,45.5,45.0,47.33
RIDER-3152132495,RETEST,0.876,146.29,176.47,92.1,92.5,86.08
RIDER-3152132495,TEST,0.815,149.11,194.28,86.6,95.3,88.46
RIDER-3160137230,RETEST,0.731,7.3,5.03,29.5,22.9,27.41
RIDER-3160137230,TEST,0.722,8.74,5.15,25.8,22.9,27.72
RIDER-5195703382,TEST,0.585,34.99,61.89,51.7,62.6,53.18
RIDER-6324870494,RETEST,0.74,9.39,11.02,31.0,30.2,35.02
RIDER-6324870494,TEST,0.704,7.8,10.73,30.1,33.3,32.43
RIDER-8509201188,TEST,0.839,148.96,178.03,86.7,89.6,9.07
RIDER-9763310455,RETEST,0.702,0.96,1.28,15.1,18.3,18.84
RIDER-9763310455,TEST,0.643,0.85,1.29,14.1,16.6,15.46'''

REF_CSV = '''patient,scan,dice_auto_vs_expert
RIDER-1129164940,TEST,0.915
RIDER-1129164940,RETEST,0.97
RIDER-1225316081,TEST,0.939
RIDER-1225316081,RETEST,0.758
RIDER-1286684383,TEST,0.662
RIDER-1286684383,RETEST,0.714
RIDER-1332496276,TEST,0.977
RIDER-1332496276,RETEST,0.977
RIDER-1344011628,TEST,0.9
RIDER-1344011628,RETEST,0.96
RIDER-1346917136,TEST,0.969
RIDER-1346917136,RETEST,0.982
RIDER-1500037140,TEST,0.988
RIDER-1500037140,RETEST,0.959
RIDER-1532432635,RETEST,0.649
RIDER-1532432635,TEST,0.73
RIDER-1760553574,RETEST,0.486
RIDER-1760553574,TEST,0.416
RIDER-1801720707,TEST,0.633
RIDER-1801720707,RETEST,0.798
RIDER-1825099523,RETEST,0.974
RIDER-1825099523,TEST,0.969
RIDER-1867199702,TEST,0.928
RIDER-1867199702,RETEST,0.895
RIDER-2016615262,TEST,0.594
RIDER-2016615262,RETEST,
RIDER-2073083717,RETEST,0.956
RIDER-2073083717,TEST,0.965
RIDER-2151469008,RETEST,0.951
RIDER-2151469008,TEST,0.836
RIDER-2283289298,RETEST,0.975
RIDER-2357766186,RETEST,0.881
RIDER-2357766186,TEST,0.83
RIDER-2539508879,RETEST,0.967
RIDER-2539508879,TEST,0.93
RIDER-2541949645,TEST,0.725
RIDER-2541949645,RETEST,0.788
RIDER-2619750334,RETEST,0.903
RIDER-2619750334,TEST,0.846
RIDER-2655999012,TEST,0.849
RIDER-2655999012,RETEST,0.871
RIDER-2669524182,RETEST,0.932
RIDER-2669524182,TEST,0.839
RIDER-2799584460,RETEST,0.981
RIDER-2799584460,TEST,0.995
RIDER-3023568408,TEST,0.937
RIDER-3023568408,RETEST,0.879
RIDER-3115188676,TEST,0.828
RIDER-3115188676,RETEST,0.987
RIDER-3152132495,RETEST,0.96
RIDER-3152132495,TEST,0.92
RIDER-3160137230,TEST,0.836
RIDER-3160137230,RETEST,0.884
RIDER-5195703382,TEST,0.92
RIDER-6324870494,TEST,0.988
RIDER-6324870494,RETEST,0.958
RIDER-8509201188,TEST,0.942
RIDER-9763310455,TEST,0.886
RIDER-9763310455,RETEST,0.924'''

## 3. Download the scans and expert outlines from TCIA (public API, no login)

In [ ]:
import os, io, zipfile, requests, concurrent.futures as cf
API = "https://services.cancerimagingarchive.net/nbia-api/services/v1/getImage?SeriesInstanceUID="
DATA = "/content/rider"

def fetch(uid):
    out = f"{DATA}/{uid}"
    if os.path.isdir(out) and os.listdir(out):
        return uid, "cached"
    for attempt in range(4):
        try:
            r = requests.get(API + uid, timeout=600)
            r.raise_for_status()
            os.makedirs(out, exist_ok=True)
            zipfile.ZipFile(io.BytesIO(r.content)).extractall(out)
            return uid, "ok"
        except Exception as e:
            err = e
    return uid, f"FAILED {err}"

uids = sorted({c[2] for c in cases} | {c[3] for c in cases})
with cf.ThreadPoolExecutor(6) as ex:
    for i, (uid, status) in enumerate(ex.map(fetch, uids), 1):
        if status != "cached":
            print(f"{i}/{len(uids)} {status}")
print("done:", len(uids), "series")

## 4. Same helper code as our engine: read the scan, the expert outline, and the simulated box

In [ ]:
import glob, numpy as np, pydicom, SimpleITK as sitk

def load_ct(uid):
    d = f"{DATA}/{uid}"
    r = sitk.ImageSeriesReader(); r.SetFileNames(r.GetGDCMSeriesFileNames(d))
    return r.Execute()

def expert_mask(seg_uid, ct, label="MANUAL"):
    f = glob.glob(f"{DATA}/{seg_uid}/*.dcm")[0]
    ds = pydicom.dcmread(f)
    seg_no = next(s.SegmentNumber for s in ds.SegmentSequence if s.SegmentAlgorithmType == label)
    frames = ds.pixel_array
    frames = frames[None] if frames.ndim == 2 else frames
    nz = ct.GetSize()[2]
    zs = np.array([ct.TransformIndexToPhysicalPoint((0, 0, k))[2] for k in range(nz)])
    m = np.zeros((nz, ct.GetSize()[1], ct.GetSize()[0]), bool)
    for i, fg in enumerate(ds.PerFrameFunctionalGroupsSequence):
        if fg.SegmentIdentificationSequence[0].ReferencedSegmentNumber != seg_no:
            continue
        z = float(fg.PlanePositionSequence[0].ImagePositionPatient[2])
        k = int(np.argmin(np.abs(zs - z)))
        if abs(zs[k] - z) < 0.5:
            m[k] |= frames[i] > 0
    return m

def box_from_expert(m):
    z = int(np.argmax(m.sum(axis=(1, 2))))
    ys, xs = np.nonzero(m[z])
    return z, (ys.min(), ys.max() + 1, xs.min(), xs.max() + 1)

def dice(a, b):
    s = a.sum() + b.sum()
    return 2 * np.logical_and(a, b).sum() / s if s else 1.0

def volume_ml(m, sp):
    return m.sum() * sp[0] * sp[1] * sp[2] / 1000

def longest_diameter_mm(m, sp):
    best, rng = 0.0, np.random.default_rng(0)
    for z in np.nonzero(m.any(axis=(1, 2)))[0]:
        ys, xs = np.nonzero(m[z]); pts = np.stack([ys * sp[1], xs * sp[0]], 1)
        if len(pts) > 2500: pts = pts[rng.choice(len(pts), 2500, replace=False)]
        best = max(best, float(np.sqrt(((pts[:, None] - pts[None]) ** 2).sum(-1)).max()))
    return best

## 5. MedSAM 2: one box on one slice → 3D outline (the authors' own inference recipe)

In [ ]:
from PIL import Image
from skimage import measure
import sys
sys.path.insert(0, "/content/MedSAM2")
from sam2.build_sam import build_sam2_video_predictor_npz
torch.set_float32_matmul_precision("high")
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
MEAN = torch.tensor((0.485, 0.456, 0.406), dtype=torch.float32)[:, None, None].cuda()
STD = torch.tensor((0.229, 0.224, 0.225), dtype=torch.float32)[:, None, None].cuda()

def to_frames(vol_u8, size=512):
    out = np.zeros((len(vol_u8), 3, size, size), np.float32)
    for i, s in enumerate(vol_u8):
        out[i] = np.array(Image.fromarray(s).convert("RGB").resize((size, size))).transpose(2, 0, 1)
    t = torch.from_numpy(out / 255.0).cuda()
    return (t - MEAN) / STD

def largest_cc(m):
    if m.max() == 0: return m.astype(bool)
    lab = measure.label(m)
    return lab == np.argmax(np.bincount(lab.flat)[1:]) + 1

def medsam2_box(predictor, ct_arr, z, box, half=60):
    z0, z1 = max(0, z - half), min(len(ct_arr), z + half + 1)
    sub = np.clip(ct_arr[z0:z1], *WINDOW).astype(np.float32)
    sub = np.uint8((sub - sub.min()) / max(1e-6, sub.max() - sub.min()) * 255)
    H, W = sub.shape[1:]
    y0, y1, x0, x1 = box
    bbox = np.array([x0, y0, x1 - 1, y1 - 1])
    seg = np.zeros(sub.shape, np.uint8)
    frames = to_frames(sub)
    with torch.inference_mode(), torch.autocast("cuda", dtype=AMP):
        for reverse in (False, True):
            st = predictor.init_state(frames, H, W)
            predictor.add_new_points_or_box(inference_state=st, frame_idx=z - z0, obj_id=1, box=bbox)
            for fi, _, logits in predictor.propagate_in_video(st, reverse=reverse):
                seg[fi, (logits[0] > 0.0).cpu().numpy()[0]] = 1
            predictor.reset_state(st)
    out = np.zeros(ct_arr.shape, bool)
    out[z0:z1] = largest_cc(seg)
    return out

## 6. Run the exam

In [ ]:
import time, pandas as pd
rows = []
predictors = {m: build_sam2_video_predictor_npz("configs/sam2.1_hiera_t512.yaml", f"checkpoints/{m}") for m in MODELS}
for i, (pid, scan, seg_uid, ct_uid) in enumerate(cases, 1):
    try:
        ct = load_ct(ct_uid); a = sitk.GetArrayFromImage(ct); sp = ct.GetSpacing()
        m = expert_mask(seg_uid, ct); z, box = box_from_expert(m)
        for name, pr in predictors.items():
            t = time.time(); p = medsam2_box(pr, a, z, box); torch.cuda.synchronize(); secs = time.time() - t
            rows.append(dict(patient=pid, scan=scan, model=name.replace(".pt", ""), dice=round(dice(p, m), 3),
                             expert_ml=round(volume_ml(m, sp), 2), tool_ml=round(volume_ml(p, sp), 2),
                             expert_mm=round(longest_diameter_mm(m, sp), 1), tool_mm=round(longest_diameter_mm(p, sp), 1),
                             seconds=round(secs, 2)))
            print(f"{i}/{len(cases)} {pid} {scan} {name}: Dice {rows[-1]['dice']}  ({secs:.1f}s)")
    except Exception as e:
        print(f"{i}/{len(cases)} {pid} {scan}: ERROR {e}")
res = pd.DataFrame(rows); res.to_csv("/content/medsam2_results.csv", index=False)

## 7. Compare with our engine and with the in-house tool

In [ ]:
from io import StringIO
v5 = pd.read_csv(StringIO(V5_CSV)).assign(model="our_engine_v5")
ref = pd.read_csv(StringIO(REF_CSV))
keys = res[["patient", "scan"]].drop_duplicates()
allr = pd.concat([res, v5.merge(keys, on=["patient", "scan"])], ignore_index=True)
allr["vol_err_%"] = (allr.tool_ml - allr.expert_ml).abs() / allr.expert_ml * 100
allr["diam_err_mm"] = (allr.tool_mm - allr.expert_mm).abs()
summary = allr.groupby("model").agg(
    scans=("dice", "size"), median_dice=("dice", "median"),
    share_dice_ge_080=("dice", lambda d: round((d >= 0.8).mean() * 100)),
    median_vol_err_pct=("vol_err_%", "median"), median_diam_err_mm=("diam_err_mm", "median"),
    median_seconds=("seconds", "median")).round(2)
r = ref.merge(keys, on=["patient", "scan"])["dice_auto_vs_expert"].dropna()
print("In-house auto tool (dataset authors) median Dice on the same scans:", round(r.median(), 3))
print("Expert's own scan-to-scan noise: 2.3 mm diameter, 17% volume (median)")
summary

## 8. Look at the worst, a middle and the best case (green = expert, orange = best MedSAM 2 model)

In [ ]:
import matplotlib.pyplot as plt
best = MODELS[int(np.argmax([res[res.model == m.replace('.pt','')].dice.median() for m in MODELS]))]
show = res[res.model == best.replace('.pt','')].sort_values("dice").iloc[[0, len(res)//(2*len(MODELS)), -1]]
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, (_, row) in zip(axs, show.iterrows()):
    c = next(c for c in cases if c[0] == row.patient and c[1] == row.scan)
    ct = load_ct(c[3]); a = sitk.GetArrayFromImage(ct); m = expert_mask(c[2], ct); z, box = box_from_expert(m)
    p = medsam2_box(predictors[best], a, z, box)
    y0, y1, x0, x1 = box; pad = 60
    sl = (slice(max(0, y0 - pad), y1 + pad), slice(max(0, x0 - pad), x1 + pad))
    ax.imshow(np.clip(a[z][sl], -1000, 400), cmap="gray")
    ax.contour(m[z][sl], [0.5], colors="lime"); ax.contour(p[z][sl], [0.5], colors="orange")
    ax.set_title(f"{row.patient} {row.scan}  Dice {row.dice}"); ax.axis("off")
plt.tight_layout(); plt.savefig("/content/medsam2_examples.png", dpi=80); plt.show()

## 9. Download the results (send these two files back)

In [ ]:
from google.colab import files
files.download("/content/medsam2_results.csv")
files.download("/content/medsam2_examples.png")